In [0]:
# COMMAND ----------

%pip install xgboost==3.0.2 scikit-learn pandas scipy mlflow

In [0]:
# COMMAND ----------

import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn

from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

from pyspark.sql import functions as F

print("Libraries imported successfully.")
print("XGBoost version:", __import__("xgboost").__version__)
print("MLflow version:", mlflow.__version__)

In [0]:
# COMMAND ----------

CATALOG = "aml_engine"
SCHEMA = "aml_poc"

ML_TRAINING_TABLE = (
    f"{CATALOG}.{SCHEMA}.ml_training_data"
)

REGISTERED_MODEL_NAME = (
    f"{CATALOG}.{SCHEMA}.aml_xgboost_model"
)

TARGET_COLUMN = "label"

RANDOM_SEED = 42

# Initial threshold only.
# This will NOT be the final threshold.
INITIAL_THRESHOLD = 0.50

# Training sample:
# Keep all fraud records.
# Keep at most 20 normal records for each fraud record.
NEGATIVE_TO_POSITIVE_RATIO = 20

print("Training table:", ML_TRAINING_TABLE)
print("UC model:", REGISTERED_MODEL_NAME)

In [0]:
# COMMAND ----------

training_df = spark.table(
    ML_TRAINING_TABLE
)

total_records = training_df.count()

print("Total records:", total_records)

display(
    training_df.limit(10)
)

In [0]:
# COMMAND ----------

display(
    training_df
    .groupBy("label")
    .count()
    .orderBy("label")
)

In [0]:
# COMMAND ----------

training_df.select(
    F.min("event_time").alias("min_event_time"),
    F.max("event_time").alias("max_event_time")
).show()

In [0]:
# COMMAND ----------

feature_columns = [
    "tx_amount",
    "tx_type",
    "event_time",

    "sender_country",
    "receiver_country",

    "sender_account_type",
    "receiver_account_type",

    "sender_init_balance",
    "receiver_init_balance",

    "sender_tx_count_before",
    "receiver_tx_count_before",

    "sender_total_amount_before",
    "receiver_total_amount_before",

    "sender_velocity_count",
    "receiver_velocity_count",

    "unique_receivers_before",
    "unique_senders_before",

    "high_value_flag",
    "velocity_flag",

    "fan_in_flag",
    "fan_out_flag",

    "fan_in_feature",
    "fan_out_feature",

    "sender_out_degree_before",
    "receiver_in_degree_before",

    "cycle_detected_as_of_time",
    "cycle_count"
]

forbidden_columns = [
    "tx_id",
    "is_fraud",
    "alert_id",
    "alert_type",
    "rule_score",
    "label"
]

leakage_columns = [
    c for c in feature_columns
    if c in forbidden_columns
]

if leakage_columns:
    raise ValueError(
        f"Forbidden columns found: {leakage_columns}"
    )

print("Feature count:", len(feature_columns))
print("Leakage check passed.")

In [0]:
# COMMAND ----------

categorical_columns = [
    "tx_type",
    "sender_country",
    "receiver_country",
    "sender_account_type",
    "receiver_account_type"
]

numeric_columns = [
    c
    for c in feature_columns
    if c not in categorical_columns
]

print("Categorical columns:")
print(categorical_columns)

print("\nNumeric columns:")
print(numeric_columns)

In [0]:
# COMMAND ----------

time_values = (
    training_df
    .select(
        F.min("event_time").alias("min_time"),
        F.max("event_time").alias("max_time")
    )
    .collect()[0]
)

min_time = float(time_values["min_time"])
max_time = float(time_values["max_time"])

time_span = max_time - min_time

train_end = min_time + (
    time_span * 0.70
)

validation_end = min_time + (
    time_span * 0.85
)

print("Min event time:", min_time)
print("Max event time:", max_time)
print("Train end:", train_end)
print("Validation end:", validation_end)

In [0]:
# COMMAND ----------

train_df = (
    training_df
    .filter(
        F.col("event_time") <= train_end
    )
)

validation_df = (
    training_df
    .filter(
        (F.col("event_time") > train_end) &
        (F.col("event_time") <= validation_end)
    )
)

test_df = (
    training_df
    .filter(
        F.col("event_time") > validation_end
    )
)

print("Train:", train_df.count())
print("Validation:", validation_df.count())
print("Test:", test_df.count())

In [0]:
# COMMAND ----------

print("===== TRAIN =====")

train_df.groupBy(
    "label"
).count().orderBy(
    "label"
).show()

print("===== VALIDATION =====")

validation_df.groupBy(
    "label"
).count().orderBy(
    "label"
).show()

print("===== TEST =====")

test_df.groupBy(
    "label"
).count().orderBy(
    "label"
).show()

In [0]:
# COMMAND ----------

train_counts = (
    train_df
    .groupBy("label")
    .count()
    .collect()
)

train_count_map = {
    int(row["label"]): int(row["count"])
    for row in train_counts
}

positive_count = train_count_map.get(1, 0)
negative_count = train_count_map.get(0, 0)

if positive_count == 0:
    raise ValueError(
        "No fraud records in training dataset."
    )

desired_negative_count = (
    positive_count *
    NEGATIVE_TO_POSITIVE_RATIO
)

negative_fraction = min(
    1.0,
    desired_negative_count /
    negative_count
)

print("Fraud records:", positive_count)
print("Normal records:", negative_count)
print("Sampling fraction:", negative_fraction)

In [0]:
# COMMAND ----------

train_sampled_df = (
    train_df
    .sampleBy(
        "label",
        fractions={
            0: negative_fraction,
            1: 1.0
        },
        seed=RANDOM_SEED
    )
)

print(
    "Sampled training records:",
    train_sampled_df.count()
)

display(
    train_sampled_df
    .groupBy("label")
    .count()
    .orderBy("label")
)

In [0]:
# COMMAND ----------

train_pd = (
    train_sampled_df
    .select(
        *feature_columns,
        TARGET_COLUMN
    )
    .toPandas()
)

validation_pd = (
    validation_df
    .select(
        *feature_columns,
        TARGET_COLUMN
    )
    .toPandas()
)

test_pd = (
    test_df
    .select(
        *feature_columns,
        TARGET_COLUMN
    )
    .toPandas()
)

print("Train shape:", train_pd.shape)
print("Validation shape:", validation_pd.shape)
print("Test shape:", test_pd.shape)

In [0]:
# COMMAND ----------

X_train = train_pd[feature_columns]
y_train = train_pd[TARGET_COLUMN]

X_validation = validation_pd[feature_columns]
y_validation = validation_pd[TARGET_COLUMN]

X_test = test_pd[feature_columns]
y_test = test_pd[TARGET_COLUMN]

In [0]:
# COMMAND ----------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_columns
        ),
        (
            "numeric",
            "passthrough",
            numeric_columns
        )
    ]
)

In [0]:
# COMMAND ----------

sampled_positive = int(
    (y_train == 1).sum()
)

sampled_negative = int(
    (y_train == 0).sum()
)

if sampled_positive == 0:
    raise ValueError(
        "No positive training examples."
    )

scale_pos_weight = (
    sampled_negative /
    sampled_positive
)

print("Sampled positives:", sampled_positive)
print("Sampled negatives:", sampled_negative)
print(
    "scale_pos_weight:",
    scale_pos_weight
)

In [0]:
# COMMAND ----------

xgb_classifier = XGBClassifier(
    n_estimators=300,

    max_depth=6,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    objective="binary:logistic",

    eval_metric="aucpr",

    scale_pos_weight=float(
        scale_pos_weight
    ),

    random_state=RANDOM_SEED,

    n_jobs=-1
)

print("XGBoost classifier created.")

In [0]:
# COMMAND ----------

model_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            xgb_classifier
        )
    ]
)

print("Complete preprocessing + XGBoost pipeline created.")

In [0]:
# COMMAND ----------

model_pipeline.fit(
    X_train,
    y_train
)

print("XGBoost baseline training completed.")

In [0]:
# COMMAND ----------

validation_probability = (
    model_pipeline
    .predict_proba(
        X_validation
    )[:, 1]
)

print(
    "Validation predictions:",
    len(validation_probability)
)

In [0]:
# COMMAND ----------

validation_prediction_050 = (
    validation_probability >= INITIAL_THRESHOLD
).astype(int)

In [0]:
# COMMAND ----------

validation_accuracy_050 = accuracy_score(
    y_validation,
    validation_prediction_050
)

validation_precision_050 = precision_score(
    y_validation,
    validation_prediction_050,
    zero_division=0
)

validation_recall_050 = recall_score(
    y_validation,
    validation_prediction_050,
    zero_division=0
)

validation_f1_050 = f1_score(
    y_validation,
    validation_prediction_050,
    zero_division=0
)

validation_roc_auc = roc_auc_score(
    y_validation,
    validation_probability
)

validation_pr_auc = average_precision_score(
    y_validation,
    validation_probability
)

validation_cm_050 = confusion_matrix(
    y_validation,
    validation_prediction_050
)

print("======================================")
print("VALIDATION - THRESHOLD 0.50")
print("======================================")

print(
    f"Accuracy : {validation_accuracy_050:.6f}"
)

print(
    f"Precision: {validation_precision_050:.6f}"
)

print(
    f"Recall   : {validation_recall_050:.6f}"
)

print(
    f"F1       : {validation_f1_050:.6f}"
)

print(
    f"ROC-AUC  : {validation_roc_auc:.6f}"
)

print(
    f"PR-AUC   : {validation_pr_auc:.6f}"
)

print("\nConfusion Matrix:")
print(validation_cm_050)

In [0]:
# COMMAND ----------

test_probability = (
    model_pipeline
    .predict_proba(
        X_test
    )[:, 1]
)

print(
    "Test predictions:",
    len(test_probability)
)

In [0]:
# COMMAND ----------

test_prediction_050 = (
    test_probability >= INITIAL_THRESHOLD
).astype(int)

test_accuracy_050 = accuracy_score(
    y_test,
    test_prediction_050
)

test_precision_050 = precision_score(
    y_test,
    test_prediction_050,
    zero_division=0
)

test_recall_050 = recall_score(
    y_test,
    test_prediction_050,
    zero_division=0
)

test_f1_050 = f1_score(
    y_test,
    test_prediction_050,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    y_test,
    test_probability
)

test_pr_auc = average_precision_score(
    y_test,
    test_probability
)

test_cm_050 = confusion_matrix(
    y_test,
    test_prediction_050
)

print("======================================")
print("TEST - THRESHOLD 0.50")
print("======================================")

print(
    f"Accuracy : {test_accuracy_050:.6f}"
)

print(
    f"Precision: {test_precision_050:.6f}"
)

print(
    f"Recall   : {test_recall_050:.6f}"
)

print(
    f"F1       : {test_f1_050:.6f}"
)

print(
    f"ROC-AUC  : {test_roc_auc:.6f}"
)

print(
    f"PR-AUC   : {test_pr_auc:.6f}"
)

print("\nConfusion Matrix:")
print(test_cm_050)

In [0]:
# COMMAND ----------

threshold_results = []

for threshold in np.arange(
    0.05,
    1.00,
    0.01
):

    validation_prediction = (
        validation_probability >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation,
        validation_prediction,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        validation_prediction,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        validation_prediction,
        zero_division=0
    )

    threshold_results.append({
        "threshold": round(
            float(threshold),
            2
        ),
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(
    threshold_results
)

display(
    threshold_df
    .sort_values(
        "threshold"
    )
)

In [0]:
# COMMAND ----------

recall_candidates = (
    threshold_df[
        threshold_df["recall"] >= 0.90
    ]
    .sort_values(
        ["precision", "f1"],
        ascending=[False, False]
    )
)

if recall_candidates.empty:
    raise ValueError(
        "No threshold achieves the required 90% validation recall."
    )

selected_row = recall_candidates.iloc[0]

SELECTED_THRESHOLD = float(
    selected_row["threshold"]
)

SELECTED_VALIDATION_PRECISION = float(
    selected_row["precision"]
)

SELECTED_VALIDATION_RECALL = float(
    selected_row["recall"]
)

SELECTED_VALIDATION_F1 = float(
    selected_row["f1"]
)

print("======================================")
print("SELECTED THRESHOLD")
print("======================================")

print(
    "Threshold:",
    SELECTED_THRESHOLD
)

print(
    "Validation precision:",
    SELECTED_VALIDATION_PRECISION
)

print(
    "Validation recall:",
    SELECTED_VALIDATION_RECALL
)

print(
    "Validation F1:",
    SELECTED_VALIDATION_F1
)

In [0]:
# COMMAND ----------

final_test_prediction = (
    test_probability >= SELECTED_THRESHOLD
).astype(int)

In [0]:
# COMMAND ----------

final_test_accuracy = accuracy_score(
    y_test,
    final_test_prediction
)

final_test_precision = precision_score(
    y_test,
    final_test_prediction,
    zero_division=0
)

final_test_recall = recall_score(
    y_test,
    final_test_prediction,
    zero_division=0
)

final_test_f1 = f1_score(
    y_test,
    final_test_prediction,
    zero_division=0
)

final_test_cm = confusion_matrix(
    y_test,
    final_test_prediction
)

print("======================================")
print(
    f"FINAL TEST @ THRESHOLD "
    f"{SELECTED_THRESHOLD:.2f}"
)
print("======================================")

print(
    f"Accuracy  : {final_test_accuracy:.6f}"
)

print(
    f"Precision : {final_test_precision:.6f}"
)

print(
    f"Recall    : {final_test_recall:.6f}"
)

print(
    f"F1        : {final_test_f1:.6f}"
)

print(
    f"ROC-AUC   : {test_roc_auc:.6f}"
)

print(
    f"PR-AUC    : {test_pr_auc:.6f}"
)

print("\nConfusion Matrix:")
print(final_test_cm)

In [0]:
# COMMAND ----------

tn, fp, fn, tp = final_test_cm.ravel()

actual_fraud = int(
    (y_test == 1).sum()
)

predicted_fraud = int(
    (final_test_prediction == 1).sum()
)

print("======================================")
print("FINAL AML TEST SUMMARY")
print("======================================")

print("Actual fraud transactions :", actual_fraud)
print("Detected fraud            :", tp)
print("Missed fraud              :", fn)

print("True negatives            :", tn)
print("False positives           :", fp)

print("Generated AML alerts      :", predicted_fraud)

In [0]:
# COMMAND ----------

mlflow.set_registry_uri(
    "databricks-uc"
)

mlflow.set_experiment(
    "/Shared/AML_POC_XGBoost"
)

print("MLflow configured.")

In [0]:
# COMMAND ----------

from mlflow.models import infer_signature

signature = infer_signature(
    X_train.head(10),
    model_pipeline.predict_proba(
        X_train.head(10)
    )
)

print(signature)

In [0]:
# COMMAND ----------

if mlflow.active_run() is not None:
    print(
        "Ending active run:",
        mlflow.active_run().info.run_id
    )
    mlflow.end_run()

print(
    "Active run:",
    mlflow.active_run()
)

In [0]:
# COMMAND ----------

with mlflow.start_run(
    run_name="aml_xgboost_final"
) as final_run:

    # ==========================================
    # Parameters
    # ==========================================

    mlflow.log_param(
        "model_type",
        "XGBoost"
    )

    mlflow.log_param(
        "n_estimators",
        300
    )

    mlflow.log_param(
        "max_depth",
        6
    )

    mlflow.log_param(
        "learning_rate",
        0.05
    )

    mlflow.log_param(
        "subsample",
        0.8
    )

    mlflow.log_param(
        "colsample_bytree",
        0.8
    )

    mlflow.log_param(
        "negative_to_positive_ratio",
        NEGATIVE_TO_POSITIVE_RATIO
    )

    mlflow.log_param(
        "scale_pos_weight",
        float(scale_pos_weight)
    )

    mlflow.log_param(
        "classification_threshold",
        float(SELECTED_THRESHOLD)
    )

    mlflow.log_param(
        "feature_count",
        len(feature_columns)
    )

    mlflow.log_param(
        "target_column",
        TARGET_COLUMN
    )

    # ==========================================
    # Validation metrics
    # ==========================================

    mlflow.log_metric(
        "validation_pr_auc",
        float(validation_pr_auc)
    )

    mlflow.log_metric(
        "validation_roc_auc",
        float(validation_roc_auc)
    )

    mlflow.log_metric(
        "validation_precision_selected_threshold",
        SELECTED_VALIDATION_PRECISION
    )

    mlflow.log_metric(
        "validation_recall_selected_threshold",
        SELECTED_VALIDATION_RECALL
    )

    mlflow.log_metric(
        "validation_f1_selected_threshold",
        SELECTED_VALIDATION_F1
    )

    # ==========================================
    # Test metrics
    # ==========================================

    mlflow.log_metric(
        "test_pr_auc",
        float(test_pr_auc)
    )

    mlflow.log_metric(
        "test_roc_auc",
        float(test_roc_auc)
    )

    mlflow.log_metric(
        "test_accuracy",
        float(final_test_accuracy)
    )

    mlflow.log_metric(
        "test_precision",
        float(final_test_precision)
    )

    mlflow.log_metric(
        "test_recall",
        float(final_test_recall)
    )

    mlflow.log_metric(
        "test_f1",
        float(final_test_f1)
    )

    mlflow.log_metric(
        "test_true_positives",
        float(tp)
    )

    mlflow.log_metric(
        "test_false_positives",
        float(fp)
    )

    mlflow.log_metric(
        "test_false_negatives",
        float(fn)
    )

    mlflow.log_metric(
        "test_true_negatives",
        float(tn)
    )

    # ==========================================
    # Log model WITH signature
    # ==========================================

    model_info = mlflow.sklearn.log_model(
        sk_model=model_pipeline,
        name="aml_xgboost_model",
        signature=signature,
        input_example=X_train.head(5),
        registered_model_name=REGISTERED_MODEL_NAME
    )

    FINAL_RUN_ID = final_run.info.run_id

    print(
        "Final MLflow run:",
        FINAL_RUN_ID
    )

    print(
        "Logged model URI:",
        model_info.model_uri
    )

In [0]:
# COMMAND ----------

import mlflow

if mlflow.active_run() is not None:
    mlflow.end_run()

print("Active run:", mlflow.active_run())

In [0]:
# COMMAND ----------

from mlflow.models import infer_signature

signature = infer_signature(
    X_train.head(10),
    model_pipeline.predict_proba(
        X_train.head(10)
    )
)

print(signature)

In [0]:
# COMMAND ----------

import mlflow
import mlflow.sklearn

mlflow.set_registry_uri("databricks-uc")

mlflow.set_experiment(
    "/Shared/AML_POC_XGBoost"
)

REGISTERED_MODEL_NAME = (
    "aml_engine.aml_poc.aml_xgboost_model"
)

with mlflow.start_run(
    run_name="aml_xgboost_final"
) as final_run:

    FINAL_RUN_ID = final_run.info.run_id

    # ========================================================
    # PARAMETERS
    # ========================================================

    mlflow.log_param(
        "model_type",
        "XGBoost"
    )

    mlflow.log_param(
        "n_estimators",
        300
    )

    mlflow.log_param(
        "max_depth",
        6
    )

    mlflow.log_param(
        "learning_rate",
        0.05
    )

    mlflow.log_param(
        "subsample",
        0.8
    )

    mlflow.log_param(
        "colsample_bytree",
        0.8
    )

    mlflow.log_param(
        "negative_to_positive_ratio",
        NEGATIVE_TO_POSITIVE_RATIO
    )

    mlflow.log_param(
        "scale_pos_weight",
        float(scale_pos_weight)
    )

    mlflow.log_param(
        "classification_threshold",
        float(SELECTED_THRESHOLD)
    )

    mlflow.log_param(
        "target_column",
        TARGET_COLUMN
    )

    mlflow.log_param(
        "feature_count",
        len(feature_columns)
    )

    # ========================================================
    # VALIDATION METRICS
    # ========================================================

    mlflow.log_metric(
        "validation_pr_auc",
        float(validation_pr_auc)
    )

    mlflow.log_metric(
        "validation_roc_auc",
        float(validation_roc_auc)
    )

    mlflow.log_metric(
        "validation_precision",
        float(SELECTED_VALIDATION_PRECISION)
    )

    mlflow.log_metric(
        "validation_recall",
        float(SELECTED_VALIDATION_RECALL)
    )

    mlflow.log_metric(
        "validation_f1",
        float(SELECTED_VALIDATION_F1)
    )

    # ========================================================
    # FINAL TEST METRICS
    # ========================================================

    mlflow.log_metric(
        "test_pr_auc",
        float(test_pr_auc)
    )

    mlflow.log_metric(
        "test_roc_auc",
        float(test_roc_auc)
    )

    mlflow.log_metric(
        "test_accuracy",
        float(final_test_accuracy)
    )

    mlflow.log_metric(
        "test_precision",
        float(final_test_precision)
    )

    mlflow.log_metric(
        "test_recall",
        float(final_test_recall)
    )

    mlflow.log_metric(
        "test_f1",
        float(final_test_f1)
    )

    mlflow.log_metric(
        "test_true_positives",
        float(tp)
    )

    mlflow.log_metric(
        "test_false_positives",
        float(fp)
    )

    mlflow.log_metric(
        "test_false_negatives",
        float(fn)
    )

    mlflow.log_metric(
        "test_true_negatives",
        float(tn)
    )

    # ========================================================
    # LOG COMPLETE PIPELINE
    # ========================================================

    model_info = mlflow.sklearn.log_model(
        sk_model=model_pipeline,

        name="aml_xgboost_model",

        signature=signature,

        input_example=X_train.head(5),

        serialization_format="skops",

        skops_trusted_types=[
            "sklearn.compose._column_transformer._RemainderColsList",
            "xgboost.core.Booster",
            "xgboost.sklearn.XGBClassifier"
        ]
    )

    print("Final run ID:", FINAL_RUN_ID)
    print("Model URI:", model_info.model_uri)